# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为展示你对 **API 调用** 与本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如电力工程里的 `cos φ = 1`）
- **输出**：清晰、带解释的回答
- **本笔记本路径**：云端用 **Anthropic Claude**（流式），本地再用 **Llama 3.2** 对比

这是你在课程期间自己也能天天用的工具：遇到看不懂的概念，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 环境变量与密钥 | `ANTHROPIC_API_KEY` + `load_dotenv` |
| 流式输出 | Anthropic `stream=True`，按 `content_block_delta` 打印 |
| OpenAI 兼容本地端点 | `OpenAI(base_url=OLLAMA_BASE_URL, ...)` |

## 怎么跑

1. `.env` 配置 `ANTHROPIC_API_KEY`；本地需 Ollama 已拉取 `llama3.2`
2. 把导入格里的 `sys.path.append(...)` 改成你本机 `llm_engineering` 仓库路径（若需要）
3. 在「提问」格改写 `question`，再分别跑 Claude 与 Llama 两格


In [ ]:
# ========== 导入：后面要用的工具箱 ==========

# 导入标准库 os：读环境变量（例如 ANTHROPIC_API_KEY）
import os
# 从 anthropic 导入 Anthropic 客户端：调用 Claude Messages API
from anthropic import Anthropic
# 从 openai 导入 OpenAI 客户端：后面用来打本地 Ollama 的兼容端点
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境，避免写进代码
from dotenv import load_dotenv
# 导入 sys：用于修改模块搜索路径
import sys
# 把课程仓库路径加入 path；占位字符串保持原样——请改成你本机真实路径后再跑依赖本地模块的代码
sys.path.append(r"put_your_actual_path_here")  # Replace with the actual path to the llm_engineering repository


In [ ]:
# ========== 常量 + Anthropic 客户端 ==========

# 云端 Claude 模型 id（字符串必须保持原样）
MODEL = 'claude-opus-4-8'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2
MODEL_LLAMA = 'llama3.2'
# 创建 Anthropic 客户端：默认从环境变量读 ANTHROPIC_API_KEY
client = Anthropic()


In [ ]:
# ========== 环境：加载并粗检 ANTHROPIC_API_KEY ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 Anthropic 密钥（不要把真实密钥写进笔记本）
api_key = os.getenv('ANTHROPIC_API_KEY')

# 分层检查：缺失 / 前缀不像 sk-ant- / 首尾空白 / 看起来正常（提示文案保持英文）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-ant-"):
    print("An API key was found, but it doesn't start sk-ant-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 发给模型的问题保持英文（可运行 / 影响回答的字符串不翻译）
question = """
Please explain what does cos fi=1 mean in the power generators.
"""


In [ ]:
# ========== Claude 流式回答：边生成边打印 ==========

# messages.create：Anthropic Messages API；stream=True 表示流式事件
response = client.messages.create(
    model=MODEL,
    # 限制本次最多生成的 token 数
    max_tokens=1024,
    # system 指令保持英文：设定「老教师讲解」人设
    system="You are an old teacher that explains everything.",
    messages=[
        # user：真正的问题
        {"role": "user", "content": question}
    ],
    stream=True
)
# 迭代流式事件：只在文本增量类型时打印
for event in response:
    if event.type == "content_block_delta":
        # end="" / flush=True：同一行连续吐字，不攒缓冲
        print(event.delta.text, end="", flush=True)


In [ ]:
# ========== 对比：用本地 Llama 3.2 回答同一问题 ==========

# Ollama 的 OpenAI 兼容 Base URL
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 创建指向本地的客户端；api_key 对 Ollama 通常任意非空
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# 非流式一次拿齐回复；model 用常量 MODEL_LLAMA
response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=[{"role": "user", "content": question}])

# 打印助手完整文本，便于和上一格 Claude 流式输出对比
print(response.choices[0].message.content)
